# P&L extract — mirror of `scripts/extract_pnl_directory.py`

Use this notebook to debug workbooks (sheet names, headers, errors) before running the CLI.

**Tip:** If a file fails with `no sheet found`, inspect **Cell: list sheet names** and add the tab name to `sheet_name_aliases` in `config/pnl_extract.json`.

In [7]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

# Repo root = parent of this notebook’s folder (adjust if you opened the notebook elsewhere)
REPO_ROOT = Path(".").resolve()
SCRIPTS = REPO_ROOT / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import extract_pnl_directory as epd

CONFIG_PATH = REPO_ROOT / "config" / "igate_key_metrics_extract.json"
INPUT_DIR = REPO_ROOT / ".." / "pnl-data" / "new_format"  # change to your folder

print("REPO_ROOT:", REPO_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)
print("INPUT_DIR:", INPUT_DIR.resolve())

REPO_ROOT: /Users/kai/Work/iNova/inova/inova-pnl-extract
CONFIG_PATH: /Users/kai/Work/iNova/inova/inova-pnl-extract/config/igate_key_metrics_extract.json
INPUT_DIR: /Users/kai/Work/iNova/inova/pnl-data/new_format


In [5]:
cfg = epd.load_config(str(CONFIG_PATH))
cfg["sheet_name"], cfg.get("sheet_name_aliases", [])

('Project Summary', [])

## List sheet names (pick one for `sheet_name` / `sheet_name_aliases`)

In [8]:
SAMPLE = INPUT_DIR / "iNova LOCAL Project BLANK TEMPLATE_TEST3.xlsm"  # change filename
engine = cfg.get("excel", {}).get("engine", "openpyxl")

if SAMPLE.is_file():
    xl = pd.ExcelFile(SAMPLE, engine=engine)
    print("Sheets:", xl.sheet_names)
else:
    print("File not found:", SAMPLE)

Sheets: ['Home Tab', 'ASSUMPTIONS', 'Project MD', 'VOL_Auto', 'SP', 'COGS', 'COGS_OTHER', 'A&P ALLOC', 'A&P TABLE', 'AP_SYN', 'DIST_AGENT', 'DIST_OUT', 'CAPEX', 'DEPR', 'SALES_CUST_ADV', 'SALES_DED_TD', 'SALES_DED_KA', 'SALES_DED_FIX', 'SALES_DED_VAR', 'EMPLOY', 'OTHER_OPEX', 'ABN', 'CANNIBAL_REV', 'CANNIBAL', 'MARKET_SH', 'SETUP INDEX', 'VOL_Pipe', 'Data Export', 'SKU Analysis (001)', 'Country Analysis (002)', 'iGATE Country', 'Dynamic P&L Rep (003)', 'iGate P&L Rep (004)', 'iGate Key Metrics (005)', 'Global MD', 'iGate Score Card (006)', 'Instructions', 'Profitability Benchmark', 'VOL_Manual', 'VOL_Select', 'REV', 'COGS2', 'Cognos_Office_Connection_Cache']


## Peek list-price block (same range as config `list_price_table`)

In [ ]:
if SAMPLE.is_file():
    sheet = cfg["sheet_name"]
    try:
        resolved = epd.resolve_sheet_name(str(SAMPLE), sheet, engine, aliases=cfg.get("sheet_name_aliases") or [])
        print("Resolved sheet:", resolved)
    except Exception as e:
        print("resolve_sheet_name failed:", repr(e))
        resolved = sheet
    lp = cfg["list_price_table"]
    raw = epd.extract_excel_table_to_df(
        str(SAMPLE),
        resolved,
        lp["start_cell"],
        lp["end_cell"],
        bool(lp.get("header_row", True)),
        engine,
    )
    display(raw.head(12))
    print("Columns:", list(raw.columns))

## Run full batch (same as CLI)

In [ ]:
if not INPUT_DIR.is_dir():
    raise SystemExit(f"INPUT_DIR is not a directory: {INPUT_DIR}")

master_df, skipped = epd.extract_all_proj_summary_aft2024(str(INPUT_DIR), cfg, verbose=True)
print("rows:", 0 if master_df is None else len(master_df))
print("skipped:", skipped)
master_df.head() if master_df is not None and not master_df.empty else None

## Single-file full extract (see exact exception if batch hides detail)

In [ ]:
if SAMPLE.is_file():
    try:
        one = epd.extract_project_summary(str(SAMPLE), cfg, verbose=True)
        display(one.head())
    except Exception as e:
        import traceback
        traceback.print_exc()